# Основы работы с Pull Requests (PR) и GitHub Actions

Этот notebook поможет вам изучить основы работы с Pull Requests, GitHub Actions и интеграцией с Gemini CLI.

## Содержание:
1. Основы Pull Requests
2. Работа с Git командами
3. GitHub Actions
4. Интеграция с Gemini CLI
5. Практические примеры

## 1. Основы Pull Requests

Pull Request (PR) - это способ предложить изменения в проект. Основные этапы:
- Fork репозитория или создание новой ветки
- Внесение изменений
- Создание PR
- Code Review
- Merge в основную ветку

In [1]:
import os
import subprocess
import json
from datetime import datetime

# Функция для выполнения git команд
def run_git_command(command):
    """Выполняет git команду и возвращает результат"""
    try:
        result = subprocess.run(command, shell=True, capture_output=True, text=True)
        return result.stdout.strip() if result.returncode == 0 else result.stderr.strip()
    except Exception as e:
        return f"Ошибка: {str(e)}"

# Проверяем статус git репозитория
def check_git_status():
    """Проверяет текущий статус git репозитория"""
    commands = {
        'branch': 'git branch --show-current',
        'status': 'git status --porcelain',
        'remote': 'git remote -v'
    }
    
    results = {}
    for key, cmd in commands.items():
        results[key] = run_git_command(cmd)
    
    return results

print("Git утилиты загружены успешно!")

Git утилиты загружены успешно!


## 2. Основные Git команды для работы с PR

In [2]:
# Демонстрация основных git команд
git_workflow_commands = {
    "Клонирование репозитория": "git clone <repository-url>",
    "Создание новой ветки": "git checkout -b feature/new-feature",
    "Просмотр изменений": "git diff",
    "Добавление файлов": "git add .",
    "Коммит изменений": "git commit -m 'Add new feature'",
    "Отправка в удаленный репозиторий": "git push origin feature/new-feature",
    "Обновление ветки": "git pull origin main",
    "Слияние веток": "git merge feature/new-feature"
}

print("=== Основные Git команды для работы с PR ===")
for description, command in git_workflow_commands.items():
    print(f"{description:.<40} {command}")

# Проверим текущий статус (если мы в git репозитории)
try:
    status = check_git_status()
    print("\n=== Текущий статус репозитория ===")
    print(f"Текущая ветка: {status['branch']}")
    print(f"Изменения: {status['status'] if status['status'] else 'Нет изменений'}")
except:
    print("\nВы не находитесь в git репозитории или git не установлен")

=== Основные Git команды для работы с PR ===
Клонирование репозитория................ git clone <repository-url>
Создание новой ветки.................... git checkout -b feature/new-feature
Просмотр изменений...................... git diff
Добавление файлов....................... git add .
Коммит изменений........................ git commit -m 'Add new feature'
Отправка в удаленный репозиторий........ git push origin feature/new-feature
Обновление ветки........................ git pull origin main
Слияние веток........................... git merge feature/new-feature

=== Текущий статус репозитория ===
Текущая ветка: prd
Изменения: M .DS_Store
 D .roo/mcp.json
 D 1.env
 D agents
 D app.py
 D "artefacts/artefact1.md\321\201"
 D bg_instruction_leila.md
 D docs/__pycache__/checko_data.cpython-312.pyc
 D docs/checko_data.py
 D docs/description.md
 D docs/optimization_plan.md
 D docs/prd/prd_from_gemini/PRD_v1_MVP.md
 D docs/prd/prd_from_gemini/PRD_v2_Broker_Pro.md
 D docs/prd/prd_from_gemi

## 3. GitHub Actions - Автоматизация CI/CD

GitHub Actions позволяют автоматизировать рабочие процессы:

In [3]:
# Создание файла GitHub Actions workflow
def create_github_action_workflow():
    """Создает базовый GitHub Actions workflow файл"""
    
    workflow_content = '''name: CI/CD Pipeline

on:
  push:
    branches: [ main, develop ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest
    
    steps:
    - uses: actions/checkout@v3
    
    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.9'
    
    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt
    
    - name: Run tests
      run: |
        python -m pytest tests/
    
    - name: Run linting
      run: |
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

  gemini-review:
    runs-on: ubuntu-latest
    if: github.event_name == 'pull_request'
    
    steps:
    - uses: actions/checkout@v3
    
    - name: Run Gemini CLI Review
      uses: google-github-actions/run-gemini-cli@v1
      with:
        command: 'review'
        files: ${{ github.event.pull_request.changed_files }}
      env:
        GEMINI_API_KEY: ${{ secrets.GEMINI_API_KEY }}'''
    
    # Создаем директорию .github/workflows если её нет
    os.makedirs('.github/workflows', exist_ok=True)
    
    # Записываем workflow файл
    with open('.github/workflows/ci.yml', 'w') as f:
        f.write(workflow_content)
    
    print("✅ GitHub Actions workflow создан: .github/workflows/ci.yml")
    return workflow_content

# Создаем workflow
workflow = create_github_action_workflow()
print("\n=== Содержимое GitHub Actions workflow ===")
print(workflow)

✅ GitHub Actions workflow создан: .github/workflows/ci.yml

=== Содержимое GitHub Actions workflow ===
name: CI/CD Pipeline

on:
  push:
    branches: [ main, develop ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
    - uses: actions/checkout@v3

    - name: Set up Python
      uses: actions/setup-python@v4
      with:
        python-version: '3.9'

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements.txt

    - name: Run tests
      run: |
        python -m pytest tests/

    - name: Run linting
      run: |
        flake8 . --count --select=E9,F63,F7,F82 --show-source --statistics

  gemini-review:
    runs-on: ubuntu-latest
    if: github.event_name == 'pull_request'

    steps:
    - uses: actions/checkout@v3

    - name: Run Gemini CLI Review
      uses: google-github-actions/run-gemini-cli@v1
      with:
        command: 'review'
        files: ${{ github.event

## 4. Интеграция с Gemini CLI

Gemini CLI может помочь в автоматическом анализе кода и создании PR.

In [4]:
# Функции для работы с Gemini CLI
def setup_gemini_cli():
    """Инструкции по настройке Gemini CLI"""
    setup_instructions = {
        "1. Установка Gemini CLI": "npm install -g @google/gemini-cli",
        "2. Настройка GitHub Integration": "gemini setup-github",
        "3. Настройка API ключа": "export GEMINI_API_KEY=your_api_key",
        "4. Инициализация в проекте": "gemini init",
        "5. Запуск анализа кода": "gemini review --files=src/"
    }
    
    return setup_instructions

def create_gemini_config():
    """Создает конфигурационный файл для Gemini CLI"""
    config = {
        "version": "1.0",
        "github": {
            "auto_review": True,
            "review_on_pr": True,
            "comment_style": "constructive"
        },
        "analysis": {
            "check_security": True,
            "check_performance": True,
            "check_best_practices": True,
            "languages": ["python", "javascript", "typescript"]
        },
        "output": {
            "format": "markdown",
            "include_suggestions": True
        }
    }
    
    with open('.gemini.json', 'w') as f:
        json.dump(config, f, indent=2)
    
    print("✅ Gemini CLI конфигурация создана: .gemini.json")
    return config

# Показываем инструкции по настройке
instructions = setup_gemini_cli()
print("=== Настройка Gemini CLI ===")
for step, command in instructions.items():
    print(f"{step}: {command}")

# Создаем конфигурацию
config = create_gemini_config()
print("\n=== Конфигурация Gemini CLI ===")
print(json.dumps(config, indent=2))

=== Настройка Gemini CLI ===
1. Установка Gemini CLI: npm install -g @google/gemini-cli
2. Настройка GitHub Integration: gemini setup-github
3. Настройка API ключа: export GEMINI_API_KEY=your_api_key
4. Инициализация в проекте: gemini init
5. Запуск анализа кода: gemini review --files=src/
✅ Gemini CLI конфигурация создана: .gemini.json

=== Конфигурация Gemini CLI ===
{
  "version": "1.0",
  "github": {
    "auto_review": true,
    "review_on_pr": true,
    "comment_style": "constructive"
  },
  "analysis": {
    "check_security": true,
    "check_performance": true,
    "check_best_practices": true,
    "languages": [
      "python",
      "javascript",
      "typescript"
    ]
  },
  "output": {
    "format": "markdown",
    "include_suggestions": true
  }
}


## 5. Практический пример: Создание PR с автоматическими проверками

In [5]:
# Симуляция процесса создания PR
class PRManager:
    def __init__(self, repo_name):
        self.repo_name = repo_name
        self.current_branch = "main"
        self.prs = []
    
    def create_feature_branch(self, branch_name):
        """Создает новую ветку для разработки"""
        print(f"🔀 Создана новая ветка: {branch_name}")
        self.current_branch = branch_name
        return f"git checkout -b {branch_name}"
    
    def add_changes(self, files):
        """Добавляет изменения в файлы"""
        print(f"📝 Изменены файлы: {', '.join(files)}")
        return f"git add {' '.join(files)}"
    
    def commit_changes(self, message):
        """Создает коммит с изменениями"""
        print(f"💾 Коммит: {message}")
        return f"git commit -m '{message}'"
    
    def push_branch(self):
        """Отправляет ветку в удаленный репозиторий"""
        print(f"🚀 Отправка ветки {self.current_branch} в удаленный репозиторий")
        return f"git push origin {self.current_branch}"
    
    def create_pr(self, title, description, assignees=None):
        """Создает Pull Request"""
        pr = {
            'id': len(self.prs) + 1,
            'title': title,
            'description': description,
            'branch': self.current_branch,
            'status': 'open',
            'created_at': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
            'assignees': assignees or [],
            'checks': {
                'tests': 'pending',
                'linting': 'pending',
                'gemini_review': 'pending'
            }
        }
        
        self.prs.append(pr)
        print(f"✅ PR #{pr['id']} создан: {title}")
        return pr
    
    def run_checks(self, pr_id):
        """Запускает автоматические проверки для PR"""
        pr = self.prs[pr_id - 1]
        
        checks = {
            'tests': '✅ passed',
            'linting': '✅ passed',
            'gemini_review': '✅ approved'
        }
        
        pr['checks'] = checks
        print(f"🔍 Проверки для PR #{pr_id} завершены:")
        for check, status in checks.items():
            print(f"  {check}: {status}")
    
    def get_pr_status(self, pr_id):
        """Получает статус PR"""
        pr = self.prs[pr_id - 1]
        return pr

# Демонстрация работы с PR
print("=== Демонстрация создания PR ===")
pr_manager = PRManager("my-awesome-project")

# 1. Создаем ветку для новой функции
cmd1 = pr_manager.create_feature_branch("feature/user-authentication")

# 2. Добавляем изменения
cmd2 = pr_manager.add_changes(["src/auth.py", "tests/test_auth.py", "README.md"])

# 3. Коммитим изменения
cmd3 = pr_manager.commit_changes("Add user authentication system")

# 4. Отправляем ветку
cmd4 = pr_manager.push_branch()

# 5. Создаем PR
pr = pr_manager.create_pr(
    title="Add user authentication system",
    description="This PR adds a complete user authentication system with login, registration, and password reset functionality.",
    assignees=["reviewer1", "reviewer2"]
)

# 6. Запускаем автоматические проверки
pr_manager.run_checks(pr['id'])

# 7. Показываем финальный статус
final_status = pr_manager.get_pr_status(pr['id'])
print("\n=== Финальный статус PR ===")
print(json.dumps(final_status, indent=2, ensure_ascii=False))

=== Демонстрация создания PR ===
🔀 Создана новая ветка: feature/user-authentication
📝 Изменены файлы: src/auth.py, tests/test_auth.py, README.md
💾 Коммит: Add user authentication system
🚀 Отправка ветки feature/user-authentication в удаленный репозиторий
✅ PR #1 создан: Add user authentication system
🔍 Проверки для PR #1 завершены:
  tests: ✅ passed
  linting: ✅ passed
  gemini_review: ✅ approved

=== Финальный статус PR ===
{
  "id": 1,
  "title": "Add user authentication system",
  "description": "This PR adds a complete user authentication system with login, registration, and password reset functionality.",
  "branch": "feature/user-authentication",
  "status": "open",
  "created_at": "2025-08-09 10:10:06",
  "assignees": [
    "reviewer1",
    "reviewer2"
  ],
  "checks": {
    "tests": "✅ passed",
    "linting": "✅ passed",
    "gemini_review": "✅ approved"
  }
}


## 6. Создание файлов для проекта

In [6]:
# Создание дополнительных файлов для полноценного проекта
def create_project_files():
    """Создает необходимые файлы для проекта"""
    
    # requirements.txt
    requirements = '''pytest>=7.0.0
flake8>=4.0.0
black>=22.0.0
requests>=2.28.0'''
    
    # .gitignore
    gitignore = '''__pycache__/
*.py[cod]
*$py.class
*.so
.Python
build/
develop-eggs/
dist/
downloads/
eggs/
.eggs/
lib/
lib64/
parts/
sdist/
var/
wheels/
*.egg-info/
.installed.cfg
*.egg
MANIFEST
.env
.venv
env/
venv/
ENV/
env.bak/
venv.bak/
.idea/
.vscode/'''
    
    # README.md
    readme = '''# My Awesome Project

Проект для изучения основ работы с Pull Requests и GitHub Actions.

## Установка

```bash
pip install -r requirements.txt
```

## Запуск тестов

```bash
pytest tests/
```

## Работа с PR

1. Создайте новую ветку: `git checkout -b feature/your-feature`
2. Внесите изменения и коммитьте их
3. Отправьте ветку: `git push origin feature/your-feature`
4. Создайте Pull Request через GitHub интерфейс
5. Дождитесь прохождения всех проверок
6. Получите одобрение от ревьюеров
7. Смержите PR

## GitHub Actions

Проект использует GitHub Actions для:
- Автоматического запуска тестов
- Проверки качества кода (linting)
- Автоматического ревью с помощью Gemini CLI'''
    
    # Создаем файлы
    files_to_create = {
        'requirements.txt': requirements,
        '.gitignore': gitignore,
        'README.md': readme
    }
    
    for filename, content in files_to_create.items():
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(content)
        print(f"✅ Создан файл: {filename}")
    
    # Создаем директории
    directories = ['src', 'tests', 'docs']
    for dir_name in directories:
        os.makedirs(dir_name, exist_ok=True)
        print(f"📁 Создана директория: {dir_name}")
    
    # Создаем простой Python файл для тестирования
    sample_code = '''def add_numbers(a, b):
    """Функция для сложения двух чисел"""
    return a + b

def multiply_numbers(a, b):
    """Функция для умножения двух чисел"""
    return a * b

if __name__ == "__main__":
    print("Пример работы функций:")
    print(f"2 + 3 = {add_numbers(2, 3)}")
    print(f"4 * 5 = {multiply_numbers(4, 5)}")'''
    
    with open('src/calculator.py', 'w', encoding='utf-8') as f:
        f.write(sample_code)
    print("✅ Создан файл: src/calculator.py")
    
    # Создаем тест для этого файла
    test_code = '''import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))

from src.calculator import add_numbers, multiply_numbers

def test_add_numbers():
    """Тест функции сложения"""
    assert add_numbers(2, 3) == 5
    assert add_numbers(-1, 1) == 0
    assert add_numbers(0, 0) == 0

def test_multiply_numbers():
    """Тест функции умножения"""
    assert multiply_numbers(2, 3) == 6
    assert multiply_numbers(-1, 5) == -5
    assert multiply_numbers(0, 100) == 0'''
    
    with open('tests/test_calculator.py', 'w', encoding='utf-8') as f:
        f.write(test_code)
    print("✅ Создан файл: tests/test_calculator.py")

# Создаем все файлы проекта
create_project_files()

print("\n🎉 Проект готов! Теперь вы можете:")
print("1. Инициализировать git: git init")
print("2. Добавить удаленный репозиторий: git remote add origin <your-repo-url>")
print("3. Создать первый коммит: git add . && git commit -m 'Initial commit'")
print("4. Отправить в GitHub: git push -u origin main")
print("5. Начать работу с PR по инструкциям из README.md")

✅ Создан файл: requirements.txt
✅ Создан файл: .gitignore
✅ Создан файл: README.md
📁 Создана директория: src
📁 Создана директория: tests
📁 Создана директория: docs
✅ Создан файл: src/calculator.py
✅ Создан файл: tests/test_calculator.py

🎉 Проект готов! Теперь вы можете:
1. Инициализировать git: git init
2. Добавить удаленный репозиторий: git remote add origin <your-repo-url>
3. Создать первый коммит: git add . && git commit -m 'Initial commit'
4. Отправить в GitHub: git push -u origin main
5. Начать работу с PR по инструкциям из README.md


## 7. Полезные команды и советы

In [7]:
# Полезные команды для работы с PR
useful_commands = {
    "Работа с ветками": {
        "git branch -a": "Показать все ветки (локальные и удаленные)",
        "git checkout -b feature/name": "Создать и переключиться на новую ветку",
        "git branch -d feature/name": "Удалить локальную ветку",
        "git push origin --delete feature/name": "Удалить удаленную ветку"
    },
    "Синхронизация": {
        "git fetch origin": "Получить изменения из удаленного репозитория",
        "git pull origin main": "Получить и слить изменения из main ветки",
        "git rebase origin/main": "Перебазировать текущую ветку на main"
    },
    "Работа с коммитами": {
        "git log --oneline": "Показать историю коммитов в одну строку",
        "git commit --amend": "Изменить последний коммит",
        "git reset HEAD~1": "Отменить последний коммит (сохранив изменения)",
        "git cherry-pick <commit-hash>": "Применить коммит из другой ветки"
    },
    "GitHub CLI команды": {
        "gh pr create": "Создать PR через командную строку",
        "gh pr list": "Показать список PR",
        "gh pr checkout <number>": "Переключиться на ветку PR",
        "gh pr merge <number>": "Смержить PR"
    }
}

print("=== Полезные команды для работы с PR ===")
for category, commands in useful_commands.items():
    print(f"\n{category.upper()}:")
    for command, description in commands.items():
        print(f"  {command:<35} - {description}")

# Советы по работе с PR
print("\n=== Лучшие практики для работы с PR ===")
best_practices = [
    "Создавайте осмысленные commit сообщения",
    "Делайте PR небольшими и сфокусированными",
    "Добавляйте подробное описание к PR",
    "Используйте draft PR для незавершенной работы",
    "Проводите self-review перед созданием PR",
    "Отвечайте на комментарии ревьюеров быстро",
    "Тестируйте код локально перед push",
    "Используйте conventional commits для структурированных сообщений",
    "Обновляйте документацию при необходимости",
    "Следите за конфликтами и разрешайте их вовремя"
]

for i, practice in enumerate(best_practices, 1):
    print(f"{i}. {practice}")

=== Полезные команды для работы с PR ===

РАБОТА С ВЕТКАМИ:
  git branch -a                       - Показать все ветки (локальные и удаленные)
  git checkout -b feature/name        - Создать и переключиться на новую ветку
  git branch -d feature/name          - Удалить локальную ветку
  git push origin --delete feature/name - Удалить удаленную ветку

СИНХРОНИЗАЦИЯ:
  git fetch origin                    - Получить изменения из удаленного репозитория
  git pull origin main                - Получить и слить изменения из main ветки
  git rebase origin/main              - Перебазировать текущую ветку на main

РАБОТА С КОММИТАМИ:
  git log --oneline                   - Показать историю коммитов в одну строку
  git commit --amend                  - Изменить последний коммит
  git reset HEAD~1                    - Отменить последний коммит (сохранив изменения)
  git cherry-pick <commit-hash>       - Применить коммит из другой ветки

GITHUB CLI КОМАНДЫ:
  gh pr create                        - С